# Lesson 11: リズムとスコアファイル

**コンパニオンノートブック** — 詳しい解説は本文 Lesson 11 を参照してください。

## セットアップ

In [ ]:
# --- 最初に1回だけ実行 ---
import sys
try:
    import google.colab
    !pip install -q japanize-matplotlib
    !git clone -q https://github.com/ggszk/simple-sound-programming.git
    sys.path.append('/content/simple-sound-programming')
except ImportError:
    sys.path.append('..')

from audio_lib.notebook import setup_environment
setup_environment()

## このレッスンで学ぶこと

- テンポ（BPM）とビートの関係を数理的に理解する
- 音符の長さを秒数に変換する方法を学ぶ
- リストやデータ構造を使って音楽を表現する方法を知る
- audio_lib のシーケンサーで複数のトラックを組み合わせて楽曲を作る
- ドラムパターンをプログラムで生成する


### BPM とは

In [ ]:
bpm = 120
beat_duration = 60.0 / bpm  # 1拍の長さ（秒）
print(f"BPM {bpm} → 1拍 = {beat_duration} 秒")

### 音符の長さ

In [ ]:
def beats_to_seconds(beats, bpm):
    """拍数を秒数に変換する"""
    return beats * 60.0 / bpm

bpm = 120
print(f"4分音符: {beats_to_seconds(1, bpm)} 秒")
print(f"8分音符: {beats_to_seconds(0.5, bpm)} 秒")
print(f"付点4分音符: {beats_to_seconds(1.5, bpm)} 秒")

### 音名のリスト

In [ ]:
# きらきら星の冒頭（音名で表現）
melody_notes = ["C4", "C4", "G4", "G4", "A4", "A4", "G4"]

In [ ]:
# (音名, 拍数) のペアで表現
melody = [
    ("C4", 1), ("C4", 1), ("G4", 1), ("G4", 1),
    ("A4", 1), ("A4", 1), ("G4", 2),
]

### 休符の表現

In [ ]:
# 休符を含むメロディ
melody = [
    ("C4", 1), ("C4", 1), ("G4", 1), ("G4", 1),
    ("A4", 1), ("A4", 1), ("G4", 2),
    (None, 1),  # 1拍の休符
    ("F4", 1), ("F4", 1), ("E4", 1), ("E4", 1),
    ("D4", 1), ("D4", 1), ("C4", 2),
]

### audio_lib のシーケンサーで鳴らす

In [ ]:
from audio_lib.sequencer import Sequencer, Track, create_simple_melody
from audio_lib.instruments import Piano
from audio_lib.notebook import play_sound

# シーケンサーとトラックを準備
seq = Sequencer()
seq.tempo = 120

track = Track(name="melody", instrument=Piano())
seq.add_track(track)

# きらきら星の冒頭（MIDI ノート番号のリスト）
notes = [60, 60, 67, 67, 69, 69, 67, None, 65, 65, 64, 64, 62, 62, 60]

# メロディを追加（各音を4分音符 = 1拍で配置）
beat_sec = seq.beats_to_seconds(1)  # 1拍の秒数
create_simple_melody(track, notes, note_duration=beat_sec)

# レンダリングして再生
audio = seq.render()
play_sound(audio)

### 音符ごとに長さを変える

In [ ]:
from audio_lib.sequencer import Sequencer, Track
from audio_lib.instruments import Piano
from audio_lib.notebook import play_sound

seq = Sequencer()
seq.tempo = 120

track = Track(name="melody", instrument=Piano())
seq.add_track(track)

# (音名, 拍数) のスコアデータ
score = [
    ("C4", 1), ("C4", 1), ("G4", 1), ("G4", 1),
    ("A4", 1), ("A4", 1), ("G4", 2),
    (None, 1),
    ("F4", 1), ("F4", 1), ("E4", 1), ("E4", 1),
    ("D4", 1), ("D4", 1), ("C4", 2),
]

# スコアをトラックに変換
current_time = 0.0
for note_name, beats in score:
    duration_sec = seq.beats_to_seconds(beats)
    if note_name is not None:
        track.add_note(note_name, velocity=100,
                       start_time=current_time, duration=duration_sec)
    current_time += duration_sec

audio = seq.render()
play_sound(audio)

## 11.4 中身を見てみよう — 素の Python でスコアを処理する

In [ ]:
import numpy as np
from audio_lib.synthesis.oscillators import sine_wave
from audio_lib.synthesis.envelopes import adsr
from audio_lib.synthesis.note_utils import note_name_to_number, note_to_frequency
from audio_lib import AudioSignal
from audio_lib.notebook import play_sound

# スコアデータ
score = [
    ("C4", 1), ("C4", 1), ("G4", 1), ("G4", 1),
    ("A4", 1), ("A4", 1), ("G4", 2),
]

bpm = 120
sr = 44100

# 全体の長さを計算
total_beats = sum(beats for _, beats in score)
total_seconds = total_beats * 60.0 / bpm
total_samples = int(sr * total_seconds)

# 出力用の配列を用意
output = np.zeros(total_samples)

# スコアを1音ずつ処理
current_time = 0.0
for note_name, beats in score:
    duration = beats * 60.0 / bpm

    if note_name is not None:
        # 周波数を求める
        note_num = note_name_to_number(note_name)
        freq = note_to_frequency(note_num)

        # サイン波を生成
        t = np.linspace(0, duration, int(sr * duration), endpoint=False)
        wave = np.sin(2 * np.pi * freq * t)

        # エンベロープを適用
        env = adsr(duration, attack=0.01, decay=0.1,
                   sustain=0.7, release=0.05, sample_rate=sr)
        wave = wave * env.data

        # 出力配列に書き込む
        start = int(current_time * sr)
        end = start + len(wave)
        output[start:end] += wave

    current_time += duration

# 正規化して再生
max_val = np.max(np.abs(output))
if max_val > 0:
    output = output * 0.8 / max_val

audio = AudioSignal(output, sr)
play_sound(audio)

## 11.5 ベロシティ — 音の強さを表現する

In [ ]:
# (音名, 拍数, ベロシティ) のスコア
score = [
    ("C4", 1, 100), ("C4", 1, 80), ("G4", 1, 100), ("G4", 1, 80),
    ("A4", 1, 110), ("A4", 1, 90), ("G4", 2, 120),
]

In [ ]:
velocity = 100
amplitude = velocity / 127.0  # 0.0 〜 1.0 に変換

In [ ]:
from audio_lib.sequencer import Note

note = Note(note_number="C4", velocity=100, start_time=0.0, duration=0.5)
print(note)

### リストでドラムパターンを記述する

In [ ]:
# 8ビートの基本パターン（1 = 鳴らす、0 = 鳴らさない）
kick_pattern    = [1, 0, 0, 0, 1, 0, 0, 0]  # 1拍目と3拍目
snare_pattern   = [0, 0, 1, 0, 0, 0, 1, 0]  # 2拍目と4拍目
hihat_pattern   = [1, 1, 1, 1, 1, 1, 1, 1]  # 全ステップ

### パターンをシーケンサーに変換する

In [ ]:
from audio_lib.sequencer import Sequencer, Track
from audio_lib.instruments import Drum
from audio_lib.notebook import play_sound

seq = Sequencer()
seq.tempo = 120

drum_track = Track(name="drums", instrument=Drum())
seq.add_track(drum_track)

# パターンデータ
patterns = {
    36: [1, 0, 0, 0, 1, 0, 0, 0],  # キック
    38: [0, 0, 1, 0, 0, 0, 1, 0],  # スネア
    42: [1, 1, 1, 1, 1, 1, 1, 1],  # ハイハット
}

step_duration = seq.beats_to_seconds(0.5)  # 8分音符 = 0.5拍

# パターンを2小節分繰り返す
for repeat in range(2):
    offset = repeat * 8 * step_duration  # 小節のオフセット
    for note_num, pattern in patterns.items():
        for step, hit in enumerate(pattern):
            if hit:
                start = offset + step * step_duration
                drum_track.add_note(note_num, velocity=100,
                                    start_time=start,
                                    duration=step_duration * 0.8)

audio = seq.render()
play_sound(audio)

### パターンを変えてみる

In [ ]:
# 16ビート風のハイハット
hihat_pattern = [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]

# シャッフル風（3連符の最初と最後）
hihat_shuffle = [1, 0, 1, 1, 0, 1, 1, 0, 1, 1, 0, 1]

# ファンク風のキック
kick_funk = [1, 0, 0, 1, 1, 0, 0, 0]

## 11.7 メロディとドラムを組み合わせる

In [ ]:
from audio_lib.sequencer import Sequencer, Track
from audio_lib.instruments import Piano, Drum
from audio_lib.notebook import play_sound

seq = Sequencer()
seq.tempo = 120

# --- メロディトラック ---
melody_track = Track(name="melody", instrument=Piano())
seq.add_track(melody_track)

score = [
    ("C4", 1), ("C4", 1), ("G4", 1), ("G4", 1),
    ("A4", 1), ("A4", 1), ("G4", 2),
    (None, 1),
    ("F4", 1), ("F4", 1), ("E4", 1), ("E4", 1),
    ("D4", 1), ("D4", 1), ("C4", 2),
]

current_time = 0.0
for note_name, beats in score:
    dur = seq.beats_to_seconds(beats)
    if note_name is not None:
        melody_track.add_note(note_name, velocity=100,
                              start_time=current_time, duration=dur)
    current_time += dur

# --- ドラムトラック ---
drum_track = Track(name="drums", instrument=Drum())
drum_track.volume = 0.6  # メロディより少し控えめに
seq.add_track(drum_track)

patterns = {
    36: [1, 0, 0, 0, 1, 0, 0, 0],
    38: [0, 0, 1, 0, 0, 0, 1, 0],
    42: [1, 1, 1, 1, 1, 1, 1, 1],
}

total_steps = 32  # 4小節分（8ステップ × 4）
step_dur = seq.beats_to_seconds(0.5)

for note_num, pattern in patterns.items():
    for i in range(total_steps):
        step_in_bar = i % len(pattern)
        if pattern[step_in_bar]:
            drum_track.add_note(note_num, velocity=100,
                                start_time=i * step_dur,
                                duration=step_dur * 0.8)

# レンダリングして再生
audio = seq.render()
play_sound(audio)

### スコアファイルを読み込む関数

In [ ]:
def load_score(filename):
    """テキスト形式のスコアファイルを読み込む"""
    score = []
    with open(filename, "r") as f:
        for line in f:
            line = line.strip()
            # 空行とコメント行を飛ばす
            if not line or line.startswith("#"):
                continue
            parts = line.split()
            note_name = parts[0]
            beats = float(parts[1])
            if note_name == "R":
                note_name = None
            score.append((note_name, beats))
    return score

In [ ]:
score = load_score("twinkle.txt")
print(score[:4])